In [4]:
import sys
sys.path.insert(0, '../..')
from dependencies import *
from constants import *
from paths import *
import helper_functions

In [5]:
SUBJECTS = helper_functions.alice_get_subjects()
STIMULI = [str(i) for i in range(1, 13)]


envelopes_log = eelbrain.load.unpickle(ALICE_PROCESSED_PREDICTOR_DIR / f'~processed_envelopes-log.pickle')
durations = helper_functions.alice_get_durations(envelopes_log, STIMULI)
models = helper_functions.alice_get_models(exclude=[])

In [6]:
# LOAD EEG DATA

# Create dictionaries to hold the eeg scans of each subject, and the predictors of each model
subject_eegs = {} 
subject_model_predictors = {}

for subject in SUBJECTS:
    # Extract the raw files
    raw = mne.io.read_raw(ALICE_EEG_DIR / subject / f'{subject}_alice-raw.fif', preload=True)
    # Filter the raw data to the desired band
    raw.filter(BANDPASS_FILTER_LOW, BANDPASS_FILTER_HIGH, n_jobs=1)
    # Interpolate bad channels
    raw.interpolate_bads() # <-- No bad filters so commented out

    # Load the events embeded in the raw file
    events = eelbrain.load.fiff.events(raw)

    # Not all subjects have all trials; determine which stimuli are present
    trial_indexes = [STIMULI.index(stimulus) for stimulus in events['event']]
    # Extract the EEG data segments corresponding to the stimuli
    trial_durations = [durations[i] for i in trial_indexes]
    # Load the EEG with corresponding epochs
    eeg = eelbrain.load.fiff.variable_length_epochs(events, -0.100, 
                                                    trial_durations, decim=5) # <-- Separates by events, adds 100ms pre-stimulus, downsamples to 100Hz
    # Since trials are of unequal length, we will concatenate them for the TRF estimation.
    eeg_concatenated = eelbrain.concatenate(eeg)
    #print(eeg_concatenated)

    # Save pickle
    eelbrain.save.pickle(eeg_concatenated, ALICE_EEG_DIR / subject / f'{subject}concatenated_eeg.pickle')

    # Update dictionaries
    subject_eegs[subject] = eeg_concatenated
    subject_model_predictors[subject] = {}

    for model, predictors in models.items():
        # print(f"Concatenating: {subject} ~ {model} predictors")
        # Select and concetenate the predictors corresponding to the EEG trials
        predictors_concatenated_list = []
        for predictor in predictors:
            predictors_concatenated = eelbrain.concatenate([predictor[i] for i in trial_indexes])
            predictors_concatenated_list.append(predictors_concatenated)
        subject_model_predictors[subject][model] = predictors_concatenated_list

# Save pickle
eelbrain.save.pickle(subject_model_predictors, ALICE_PREDICTOR_DIR / f'~concatenated_predictors.pickle')
print(f'all predictors saved')


/var/folders/q5/dlf_55750jq9b22jkpyf6c940000gn/T/ipykernel_57682/3148112599.py:13: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  raw.interpolate_bads() # <-- No bad filters so commented out


all predictors saved
